# Лінійна регресія на California Housing

Мережко Олександр

Прибираю викиди й одну пов’язану ознаку, потім порівнюю результат з базовою моделлю із конспекту. Повторний EDA не проводжу, як і вказано в завданні

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import zscore
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error

pd.set_option('display.float_format', lambda x: f'{x:.4f}')

## Дані

In [2]:
housing = fetch_california_housing(as_frame=True)
data = housing.frame.copy()
print('Розмір:', data.shape)
print('Пропущених значень:', data.isna().sum().sum())
data.head()

Розмір: (20640, 9)
Пропущених значень: 0


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0000,6.9841,1.0238,322.0000,2.5556,37.8800,-122.2300,4.5260
1,8.3014,21.0000,6.2381,0.9719,2401.0000,2.1098,37.8600,-122.2200,3.5850
2,7.2574,52.0000,8.2881,1.0734,496.0000,2.8023,37.8500,-122.2400,3.5210
3,5.6431,52.0000,5.8174,1.0731,558.0000,2.5479,37.8500,-122.2500,3.4130
4,3.8462,52.0000,6.2819,1.0811,565.0000,2.1815,37.8500,-122.2500,3.4220


## Базова модель

Повторюю налаштування конспекту: 80% даних для навчання, 20% для тесту, random_state=42. Scaler навчається тільки на тренувальній вибірці. Прогноз обмежую мінімальною й максимальною ціною з навчальних даних. Ціна в наборі вимірюється в сотнях тисяч доларів

In [3]:
X_base = data.drop(columns='MedHouseVal')
y_base = data['MedHouseVal']
Xb_train, Xb_test, yb_train, yb_test = train_test_split(
    X_base, y_base, test_size=0.2, random_state=42
)

scaler_base = StandardScaler()
Xb_train_scaled = scaler_base.fit_transform(Xb_train)
Xb_test_scaled = scaler_base.transform(Xb_test)
model_base = LinearRegression().fit(Xb_train_scaled, yb_train)
pred_base = np.clip(model_base.predict(Xb_test_scaled), yb_train.min(), yb_train.max())

metrics_base = {
    'R2 train': model_base.score(Xb_train_scaled, yb_train),
    'R2 test': r2_score(yb_test, pred_base),
    'MAE': mean_absolute_error(yb_test, pred_base),
    'MAPE, %': mean_absolute_percentage_error(yb_test, pred_base) * 100,
}
pd.Series(metrics_base)

R2 train    0.6126
R2 test     0.6039
MAE         0.5215
MAPE, %    31.4299
dtype: float64

## Очистка від викидів

Викидом вважаю значення з |z| > 3. Видаляю рядок, якщо хоча б одна з чотирьох вказаних ознак перевищує цей поріг. За умовою завдання очистку виконую перед розбиттям даних

In [4]:
columns = ['AveRooms', 'AveBedrms', 'AveOccup', 'Population']
z_scores = data[columns].apply(zscore)
outliers = z_scores.abs().gt(3)
outlier_rows = outliers.any(axis=1)
clean_data = data.loc[~outlier_rows].copy()

print('Видалено рядків:', outlier_rows.sum())
print('Залишилось рядків:', len(clean_data))
outliers.sum().rename('Кількість викидів')

Видалено рядків: 505
Залишилось рядків: 20135


AveRooms      133
AveBedrms     145
AveOccup        8
Population    342
Name: Кількість викидів, dtype: int64

## Кореляція ознак

Перевіряю матрицю вихідних даних, як у конспекті. Географічні координати залишаю: вони потрібні для врахування розташування житла

In [5]:
correlation = data.drop(columns=['Latitude', 'Longitude']).corr()
correlation.round(2)

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,MedHouseVal
MedInc,1.0000,-0.1200,0.3300,-0.0600,0.0000,0.0200,0.6900
HouseAge,-0.1200,1.0000,-0.1500,-0.0800,-0.3000,0.0100,0.1100
AveRooms,0.3300,-0.1500,1.0000,0.8500,-0.0700,-0.0000,0.1500
AveBedrms,-0.0600,-0.0800,0.8500,1.0000,-0.0700,-0.0100,-0.0500
Population,0.0000,-0.3000,-0.0700,-0.0700,1.0000,0.0700,-0.0200
AveOccup,0.0200,0.0100,-0.0000,-0.0100,0.0700,1.0000,-0.0200
MedHouseVal,0.6900,0.1100,0.1500,-0.0500,-0.0200,-0.0200,1.0000


AveRooms та AveBedrms сильно пов’язані між собою. Видаляю AveBedrms, а AveRooms залишаю, оскільки її зв’язок із цільовою ціною сильніший

In [6]:
print('Кореляція AveRooms і AveBedrms:', round(correlation.loc['AveRooms', 'AveBedrms'], 3))
print('Кореляція з ціною:')
print(correlation.loc[['AveRooms', 'AveBedrms'], 'MedHouseVal'])

X = clean_data.drop(columns=['MedHouseVal', 'AveBedrms'])
y = clean_data['MedHouseVal']
print('Ознаки моделі:', X.columns.tolist())

Кореляція AveRooms і AveBedrms: 0.848
Кореляція з ціною:
AveRooms     0.1519
AveBedrms   -0.0467
Name: MedHouseVal, dtype: float64
Ознаки моделі: ['MedInc', 'HouseAge', 'AveRooms', 'Population', 'AveOccup', 'Latitude', 'Longitude']


## Оновлена модель

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LinearRegression().fit(X_train_scaled, y_train)
y_pred = np.clip(model.predict(X_test_scaled), y_train.min(), y_train.max())
metrics_updated = {
    'R2 train': model.score(X_train_scaled, y_train),
    'R2 test': r2_score(y_test, y_pred),
    'MAE': mean_absolute_error(y_test, y_pred),
    'MAPE, %': mean_absolute_percentage_error(y_test, y_pred) * 100,
}
comparison = pd.DataFrame([metrics_base, metrics_updated], index=['Базова', 'Після обробки'])
comparison

,R2 train,R2 test,MAE,"MAPE, %"
Базова,0.6126,0.6039,0.5215,31.4299
Після обробки,0.6367,0.6651,0.4999,29.3286


## Висновки

У конспекті наведено R² на навчальній вибірці 0,61, MAE 0,52 і MAPE 31%. У таблиці вище R² train відповідає цьому способу оцінювання, а R² test окремо показує результат на тесті. Нижче порівнюю фактично отримані значення

In [8]:
print(f"R² train: {metrics_base['R2 train']:.2f} → {metrics_updated['R2 train']:.2f}")
print(f"R² test: {metrics_base['R2 test']:.2f} → {metrics_updated['R2 test']:.2f}")
print(f"MAE: {metrics_base['MAE']:.2f} → {metrics_updated['MAE']:.2f}")
print(f"MAPE: {metrics_base['MAPE, %']:.2f}% → {metrics_updated['MAPE, %']:.2f}%")
print(f"Зменшення MAPE: {metrics_base['MAPE, %'] - metrics_updated['MAPE, %']:.2f} відсоткового пункта")

R² train: 0.61 → 0.64
R² test: 0.60 → 0.67
MAE: 0.52 → 0.50
MAPE: 31.43% → 29.33%
Зменшення MAPE: 2.10 відсоткового пункта


Після підготовки даних похибка зменшилася, а R² зріс. Для покращення цієї моделі вистачило обробки даних, без переходу до складнішого алгоритму. Водночас після вилучення рядків змінився й склад тестової вибірки, тому це порівняння двох варіантів підготовки, а не оцінка на однаковому тесті. Для реального застосування правила очистки варто визначати лише на навчальних даних

Джерела: [умова ДЗ](https://www.edu.goit.global/learn/43680690/23598278/23988991/homework), [приклад із конспекту](https://www.edu.goit.global/learn/43680690/23598278/23988973/training?blockId=24848656), [California Housing](https://scikit-learn.org/stable/datasets/real_world.html#california-housing-dataset)